# S&P 500 대용량 초반 분석 및 Google Drive / Supabase 연동 노트북

이 노트북은 **Google Colab의 빠른 클라우드 네트워크**를 이용해 S&P 500 전 종목(약 500개)의 장기 주가 데이터(5년~10년)를 직접 다운로드하여 **Google Drive에 Parquet 파일로 저장**하고, 41가지 매매기법 조합을 일괄 백테스팅한 뒤 **Supabase DB에 초기 기준 데이터를 자동 업로드**합니다.

### 실행 방법
1. 내 PC의 `stock-prediction` 프로젝트 폴더를 **Google Drive `내 드라이브(MyDrive)`**에 그대로 업로드합니다.
2. Google Drive에 올라간 `stock-prediction/colab_batch_trainer.ipynb` 파일을 더블클릭해 **Google Colab**으로 엽니다.
3. 상단 메뉴에서 **[런타임] -> [모두 실행]**을 누르면 구글 드라이브 마운트 후 모든 작업이 전자동으로 진행됩니다.

In [ ]:
# 1. Google Drive 마운트 및 필수 패키지 설치
from google.colab import drive
drive.mount('/content/drive')

!pip install -q yfinance pandas numpy pyarrow lxml requests supabase python-dotenv

In [ ]:
# 2. 프로젝트 경로 연결 및 데이터 저장 폴더 설정
import os
import sys
from pathlib import Path

# Google Drive에 업로드한 폴더 경로 (기본: MyDrive/stock-prediction)
PROJECT_DIR = Path('/content/drive/MyDrive/stock-prediction')
DRIVE_DATA_DIR = PROJECT_DIR / 'data' / 'colab_warehouse'
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('프로젝트 경로:', PROJECT_DIR, '| 존재 여부:', PROJECT_DIR.exists())
print('대용량 데이터 저장 폴더:', DRIVE_DATA_DIR)

In [ ]:
# 3. S&P 500 전 종목 장기 주가 데이터 다운로드 -> Google Drive에 Parquet로 압축 저장
import time
import pandas as pd
import yfinance as yf
from src.data_loader import get_sp500_constituents, normalize_ticker

PERIOD = '10y'  # 대용량 분석 기간: '5y' 또는 '10y' 또는 'max'
PARQUET_PATH = DRIVE_DATA_DIR / f'sp500_ohlcv_{PERIOD}.parquet'

constituents = get_sp500_constituents()
tickers = [normalize_ticker(t) for t in constituents['Symbol'].unique().tolist()]
print(f'대상 종목 수: {len(tickers)}개 | 다운로드 기간: {PERIOD}')

all_frames = []
batch_size = 50
for i in range(0, len(tickers), batch_size):
    batch = tickers[i:i + batch_size]
    print(f'다운로드 중: {i + 1} ~ {min(i + batch_size, len(tickers))} / {len(tickers)}')
    raw = yf.download(batch, period=PERIOD, interval='1d', group_by='ticker', auto_adjust=True, progress=False, threads=True)
    for t in batch:
        try:
            sub = raw[t].copy() if len(batch) > 1 else raw.copy()
            sub = sub[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
            if len(sub) >= 60:
                sub = sub.reset_index()
                sub.rename(columns={sub.columns[0]: 'Date'}, inplace=True)
                sub['Ticker'] = t
                all_frames.append(sub)
        except Exception:
            continue
    time.sleep(0.2)

big_df = pd.concat(all_frames, ignore_index=True)
big_df.to_parquet(PARQUET_PATH, index=False, compression='snappy')
file_mb = PARQUET_PATH.stat().st_size / (1024 * 1024)
print(f'저장 완료: {PARQUET_PATH} (총 {len(big_df):,}행, 용량 {file_mb:.2f} MB)')

In [ ]:
# 4. Drive에 저장된 대용량 Parquet 파일 불러와서 41개 매매기법 조합 전체 백테스팅 실행
from src.backtester import analyze_single_stock

loaded_df = pd.read_parquet(PARQUET_PATH)
loaded_df['Date'] = pd.to_datetime(loaded_df['Date']).dt.tz_localize(None)

HOLD_DAYS = 10
MIN_TRADES = 5

all_stats = []
grouped = loaded_df.groupby('Ticker')
total_g = len(grouped)

for idx, (ticker, t_df) in enumerate(grouped, start=1):
    if idx % 50 == 0 or idx == total_g:
        print(f'백테스팅 진행: {idx}/{total_g} ({ticker})')
    t_df = t_df.set_index('Date').sort_index()
    stats_df, _, _, _ = analyze_single_stock(
        df=t_df,
        ticker=ticker,
        hold_days=HOLD_DAYS,
        min_trades=MIN_TRADES,
        use_dynamic_tpsl=True,
    )
    if not stats_df.empty:
        all_stats.append(stats_df)

baseline_df = pd.concat(all_stats, ignore_index=True)
STATS_PATH = DRIVE_DATA_DIR / f'sp500_baseline_stats_{PERIOD}.parquet'
baseline_df.to_parquet(STATS_PATH, index=False)
print(f'전체 분석 완료! 총 {len(baseline_df):,}개 (종목 x 기법조합) 결과 저장 -> {STATS_PATH}')
baseline_df.head(10)

In [ ]:
# 5. 백테스팅 요약 결과를 Supabase에 업로드
# 프로젝트의 .env 또는 설정 파일에서 자격 증명을 자동으로 불러옵니다.
from src.feedback_db import load_saved_credentials, get_supabase_client

SUPABASE_URL, SUPABASE_KEY = load_saved_credentials()

# 환경 변수에 없을 경우 직접 입력해 주세요
if not SUPABASE_URL:
    SUPABASE_URL = ''  # 예: https://your-project.supabase.co
if not SUPABASE_KEY:
    SUPABASE_KEY = ''  # Supabase API Key 입력

sb = get_supabase_client(SUPABASE_URL, SUPABASE_KEY)

upload_df = baseline_df[[
    'Ticker', 'ComboID', '매매기법 조합', '기법 수', '발생 횟수',
    '목표 도달 확률(%)', '순수 승률(%)', '기대 수익률(%)', '종합 점수'
]].rename(columns={
    'Ticker': 'ticker',
    'ComboID': 'combo_id',
    '매매기법 조합': 'combo_name',
    '기법 수': 'num_strategies',
    '발생 횟수': 'sample_count',
    '목표 도달 확률(%)': 'hit_rate',
    '순수 승률(%)': 'win_rate',
    '기대 수익률(%)': 'expected_return',
    '종합 점수': 'composite_score',
})

records = upload_df.to_dict(orient='records')
batch_sz = 1000
print(f'Supabase 업로드 시작 (총 {len(records):,}건)...')
for i in range(0, len(records), batch_sz):
    sb.table('strategy_baseline').upsert(records[i:i + batch_sz]).execute()
print(f'Supabase strategy_baseline 테이블 업로드 완료!')